In [1]:
import pandas as pd 
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


In [2]:
#Loading The Data 

data = pd.read_csv("cleaned_data.csv")
data.head()

,VendorNumber,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Avg_PurchasePrice,Num_Products,Num_Brands,PaymentDelay,Risk
0,105,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,35.710000,1,1,43,1
1,4466,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,9.370000,3,2,45,1
2,388,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,21.320000,1,1,38,0
3,480,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,14.336467,367,81,24,0
4,516,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,8.041461,89,29,36,0


In [3]:
data.drop("InvoiceDate",axis=1,inplace=True)
data.drop("PONumber",axis=1,inplace=True)
data.drop("PODate",axis=1,inplace=True)
data.drop("PayDate",axis=1,inplace=True)
data.drop("Risk",axis=1,inplace=True)
data.drop("PaymentDelay",axis=1,inplace=True)
data.drop("VendorNumber",axis=1,inplace=True)

In [4]:
data.head()

,Quantity,Dollars,Freight,Avg_PurchasePrice,Num_Products,Num_Brands
0,6,214.26,3.47,35.710000,1,1
1,15,140.55,8.57,9.370000,3,2
2,5,106.60,4.61,21.320000,1,1
3,10100,137483.78,2935.20,14.336467,367,81
4,1935,15527.25,429.20,8.041461,89,29


In [5]:
# Split Data 

train_data,test_data = train_test_split(data,test_size=0.2,random_state=42)

In [6]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4434 entries, 2609 to 860
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Quantity           4434 non-null   int64  
 1   Dollars            4434 non-null   float64
 2   Freight            4434 non-null   float64
 3   Avg_PurchasePrice  4434 non-null   float64
 4   Num_Products       4434 non-null   int64  
 5   Num_Brands         4434 non-null   int64  
dtypes: float64(3), int64(3)
memory usage: 242.5 KB


In [7]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1109 entries, 4564 to 70
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Quantity           1109 non-null   int64  
 1   Dollars            1109 non-null   float64
 2   Freight            1109 non-null   float64
 3   Avg_PurchasePrice  1109 non-null   float64
 4   Num_Products       1109 non-null   int64  
 5   Num_Brands         1109 non-null   int64  
dtypes: float64(3), int64(3)
memory usage: 60.6 KB


In [8]:
train_data_label = train_data["Freight"]
train_data.drop("Freight",axis=1,inplace=True)
test_data_label = test_data["Freight"]
test_data.drop("Freight",axis=1,inplace=True)

In [9]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4434 entries, 2609 to 860
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Quantity           4434 non-null   int64  
 1   Dollars            4434 non-null   float64
 2   Avg_PurchasePrice  4434 non-null   float64
 3   Num_Products       4434 non-null   int64  
 4   Num_Brands         4434 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 207.8 KB


In [10]:
train_data_label.info()

<class 'pandas.core.series.Series'>
Index: 4434 entries, 2609 to 860
Series name: Freight
Non-Null Count  Dtype  
--------------  -----  
4434 non-null   float64
dtypes: float64(1)
memory usage: 69.3 KB


In [11]:
num_cols = train_data.select_dtypes(include=["int64","float64"]).columns.tolist()

In [12]:
num_cols

['Quantity', 'Dollars', 'Avg_PurchasePrice', 'Num_Products', 'Num_Brands']

In [13]:
#Num Pipeline 

general_pipeline = Pipeline([
    ("num",StandardScaler())
])


In [14]:
train_processed_data = general_pipeline.fit_transform(train_data)
test_processed_data = general_pipeline.transform(test_data)

In [15]:
#Linear Model 
lin_model = LinearRegression()
lin_model.fit(train_processed_data,train_data_label)
pred_lin_model = lin_model.predict(test_processed_data)

#RandomForest 
rand_model = RandomForestRegressor(n_estimators=200,random_state=42)
rand_model.fit(train_processed_data,train_data_label)
pred_rand_model = rand_model.predict(test_processed_data)

#XGB
xgb_model = XGBRegressor(n_estimators=200,random_state=42)
xgb_model.fit(train_processed_data,train_data_label)
pred_xgb_model = xgb_model.predict(test_processed_data)

In [16]:
#Errors

print("For Linear Regression :")
print("MAE ",mean_absolute_error(test_data_label,pred_lin_model))
print("RMSE",root_mean_squared_error(test_data_label,pred_lin_model))
print("R2 score",r2_score(test_data_label,pred_lin_model))

print("For Random FOREST Regression :")
print("MAE ",mean_absolute_error(test_data_label,pred_rand_model))
print("RMSE",root_mean_squared_error(test_data_label,pred_rand_model))
print("R2 score",r2_score(test_data_label,pred_rand_model))

print("For XGB :")
print("MAE ",mean_absolute_error(test_data_label,pred_xgb_model))
print("RMSE",root_mean_squared_error(test_data_label,pred_xgb_model))
print("R2 score",r2_score(test_data_label,pred_xgb_model))


For Linear Regression :
MAE  25.11239038698039
RMSE 123.6010844840506
R2 score 0.9704172638664532
For Random FOREST Regression :
MAE  27.928539713288025
RMSE 138.86082830323787
R2 score 0.962661808379383
For XGB :
MAE  31.754894047408115
RMSE 152.7352628242478
R2 score 0.9548276785090695


In [17]:
joblib.dump(lin_model,"freight_model.pkl")
joblib.dump(general_pipeline,"freight_model_preprocessing.pkl")

['freight_model_preprocessing.pkl']

In [18]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4434 entries, 2609 to 860
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Quantity           4434 non-null   int64  
 1   Dollars            4434 non-null   float64
 2   Avg_PurchasePrice  4434 non-null   float64
 3   Num_Products       4434 non-null   int64  
 4   Num_Brands         4434 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 207.8 KB


In [19]:
test_data.head()

,Quantity,Dollars,Avg_PurchasePrice,Num_Products,Num_Brands
4564,48,352.95,15.184815,24,3
1616,34773,225706.96,6.943917,2730,248
4861,70,634.11,17.451690,103,6
230,104,987.34,9.441111,9,4
2042,4314,31768.74,9.181274,469,45


In [20]:
test_data_label.head()

4564       1.73
1616    1196.25
4861       2.85
230        4.64
2042     152.49
Name: Freight, dtype: float64

In [21]:
sample = test_data.iloc[[18]]

processed = general_pipeline.transform(sample)

pred = lin_model.predict(processed)

print("Prediction:", pred[0])
print("Actual:", test_data_label.iloc[18])

Prediction: 2259.5536337999083
Actual: 2464.45
